# Part 3: Decision Engine — ML Models & Intelligent Alerting

**SkyGeni Sales Intelligence Challenge**

---

This notebook walks through all **4 Decision Engines** implemented in the solution:

| Engine | Technique | Question It Answers |
|--------|-----------|--------------------|
| **B** | XGBoost Classifier + SHAP | *"What factors drive wins and losses?"* |
| **C** | Prophet + XGBoost Ensemble | *"How much revenue should we expect next quarter?"* |
| **D** | Z-Score Anomaly Detection | *"Is anything behaving unusually in the pipeline?"* |
| **Bonus** | LangGraph ReAct Agent | *"Can I ask questions in plain English?"* |

Each section includes: **problem definition → code → output → interpretation**.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import xgboost as xgb
import shap

from src.data_loader import load_and_prepare_data
from src.forecasting import prepare_deal_level, prepare_weekly_series, run_forecast_pipeline

print("All imports successful.")

In [ ]:
# Load preprocessed data
df, validation = load_and_prepare_data()
print(f"Data loaded: {df.shape[0]} deals, {df.shape[1]} features")
print(f"Overall Win Rate: {df['is_won'].mean()*100:.1f}%")

---
## Engine B: Win/Loss Classifier (XGBoost + SHAP)

### Problem Definition

Standard win/loss reports tell you *that* a region underperforms, but not *why*. Is it the deal size distribution? The sales cycle? The lead source mix? 

We train a **binary classifier** to predict deal outcomes, then use **SHAP** (SHapley Additive exPlanations) to decompose predictions into feature-level contributions. This reveals the *behavioral DNA* of winning vs. losing deals.

### Feature Engineering Strategy

| Feature | Engineering | Why |
|---------|-----------|-----|
| `log_amount` | `log1p(deal_amount)` | Handles right-skewed deal sizes |
| `log_cycle` | `log1p(sales_cycle_days)` | Normalizes cycle distribution |
| `amt_vs_industry` | `deal_amount / expanding_mean(industry)` | Relative sizing within industry |
| `amt_vs_rep` | `deal_amount / expanding_mean(rep)` | Is this deal large/small for this rep? |
| `region_enc` | Label encoded | Categorical |
| `industry_enc` | Label encoded | Categorical |
| `product_type_enc` | Label encoded | Categorical |
| `lead_source_enc` | Label encoded | Categorical |
| `created_month` | Month from `created_date` | Seasonality |
| `is_quarter_end` | Binary: month ∈ {3,6,9,12} | Quarter-end dynamics |
| `days_since_start` | Days since earliest deal | Time trend |

**Leakage Prevention:** We use `expanding_mean().shift(1)` for relative features, ensuring each deal only uses information available *before* it existed.

In [ ]:
# Step 1: Prepare deal-level features
# This function performs all feature engineering described above
X_df, feature_names, cat_features, label_encoders = prepare_deal_level(df)

print(f"Feature matrix shape: {X_df.shape}")
print(f"\nFeatures ({len(feature_names)}):")
for i, f in enumerate(feature_names, 1):
    print(f"  {i:2d}. {f}")
print(f"\nTarget distribution:")
print(f"  Won:  {(X_df['target']==1).sum()} ({(X_df['target']==1).mean()*100:.1f}%)")
print(f"  Lost: {(X_df['target']==0).sum()} ({(X_df['target']==0).mean()*100:.1f}%)")

### Model Training: 5-Fold Stratified Cross-Validation

We use **Stratified K-Fold** (not random split) to ensure each fold preserves the win/loss ratio. This gives us a more reliable estimate of generalization performance.

The XGBoost model uses `scale_pos_weight` to handle class imbalance automatically.

In [ ]:
# Step 2: Train XGBoost with 5-fold Stratified CV
X = X_df[feature_names].values
y = X_df['target'].values

# Handle class imbalance
pos_count = y.sum()
neg_count = len(y) - pos_count
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

# XGBoost parameters
params = {
    'max_depth': 4,
    'learning_rate': 0.05,
    'n_estimators': 200,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss',
    'random_state': 42,
    'use_label_encoder': False
}

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
models = []

print("5-Fold Stratified Cross-Validation Results:")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    models.append(model)
    
    # Evaluate
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    auc = roc_auc_score(y_val, y_prob)
    f1 = f1_score(y_val, y_pred)
    fold_results.append({'fold': fold, 'auc': auc, 'f1': f1})
    print(f"  Fold {fold}: AUC = {auc:.3f}, F1 = {f1:.3f}")

avg_auc = np.mean([r['auc'] for r in fold_results])
avg_f1 = np.mean([r['f1'] for r in fold_results])
print(f"\n  AVERAGE: AUC = {avg_auc:.3f}, F1 = {avg_f1:.3f}")

In [ ]:
# Visualize CV Results
cv_df = pd.DataFrame(fold_results)

fig = go.Figure()
fig.add_trace(go.Bar(x=cv_df['fold'].astype(str), y=cv_df['auc'], name='AUC', marker_color='#4299e1'))
fig.add_trace(go.Bar(x=cv_df['fold'].astype(str), y=cv_df['f1'], name='F1 Score', marker_color='#48bb78'))

fig.add_hline(y=avg_auc, line_dash='dash', line_color='#4299e1', annotation_text=f'Avg AUC: {avg_auc:.3f}')
fig.add_hline(y=avg_f1, line_dash='dash', line_color='#48bb78', annotation_text=f'Avg F1: {avg_f1:.3f}')

fig.update_layout(
    title='5-Fold Cross-Validation Results',
    xaxis_title='Fold', yaxis_title='Score',
    barmode='group', template='plotly_dark', height=400,
    yaxis=dict(range=[0, 1])
)
fig.show()

### SHAP Analysis: What Drives Wins and Losses?

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions. Unlike simple feature importance (which only shows *magnitude*), SHAP shows both:
- **Direction**: Does this feature push toward win or loss?
- **Magnitude**: How strong is the effect?

We use `TreeExplainer` — optimized for tree-based models — to compute SHAP values efficiently.

In [ ]:
# Step 3: SHAP Analysis on the best fold model
# We retrain on full data for the final SHAP analysis
final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y, verbose=False)

# Compute SHAP values
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

# Mean |SHAP| per feature (global importance)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)

print("SHAP Feature Importance (Global):")
print("=" * 50)
for idx in sorted_idx[::-1]:
    print(f"  {feature_names[idx]:25s} → {mean_abs_shap[idx]:.4f}")

In [ ]:
# SHAP Bar Chart (Top 10 Features)
top_n = min(10, len(feature_names))
top_features = [feature_names[i] for i in sorted_idx[-top_n:]]
top_importance = [mean_abs_shap[i] for i in sorted_idx[-top_n:]]

fig = px.bar(
    x=top_importance, y=top_features, orientation='h',
    title='Top Win/Loss Drivers (Mean |SHAP| Value)',
    labels={'x': 'Average Impact on Prediction', 'y': ''},
    color=top_importance,
    color_continuous_scale='Bluered'
)
fig.update_layout(
    template='plotly_dark', height=500,
    coloraxis_showscale=False,
    margin=dict(l=0)
)
fig.show()

In [ ]:
# SHAP Beeswarm Plot (detailed view)
# This shows the distribution of SHAP values for each feature
shap.summary_plot(shap_values, X, feature_names=feature_names, show=True)

### Interpreting SHAP Results

The SHAP analysis reveals the **behavioral DNA** of winning deals:

1. **Deal amount relative to industry average** is a critical driver — deals that are much larger than the industry norm tend to lose more often. This aligns with our WRE metric findings.

2. **Sales cycle (log)** is highly predictive — longer cycles correlate with losses, validating our PQS stall detection approach.

3. **Temporal features** (quarter-end, month) show that timing matters — quarter-end urgency creates both opportunities and rushed, low-quality deals.

**Executive Takeaway:** Winning isn't just about the rep's skill. It's driven by **deal sizing relative to the industry norm** and the **pace of the sales process**.

---
## Engine C: Revenue Forecast (Prophet + XGBoost Ensemble)

### Problem Definition

The CRO needs to know: *"How much revenue should we expect next quarter, and how confident should we be?"*

### Hybrid Approach: Why Two Models?

| Model | Captures | Weakness |
|-------|----------|---------|
| **Prophet** | Long-term trends, quarterly seasonality, changepoints | Ignores short-term deal signals |
| **XGBoost Regressor** | Recent dynamics: lagged revenue, pipeline velocity, momentum | No seasonal decomposition; degrades for long horizons |

By **ensembling** (50/50 weighted average), we get the best of both: Prophet's stability + XGBoost's responsiveness.

### Validation: Walk-Forward Cross-Validation

We don't use random train/test splits (that would be data leakage for time series). Instead, we use **walk-forward CV**:
1. Train on weeks 1–30, test on weeks 31–34
2. Train on weeks 1–34, test on weeks 35–38
3. ...and so on

This produces realistic error estimates because the model only ever predicts *future* data.

In [ ]:
# Step 1: Aggregate into weekly time series
weekly = prepare_weekly_series(df)

print(f"Weekly time series: {len(weekly)} weeks")
print(f"Date range: {weekly['week_start'].min()} → {weekly['week_start'].max()}")
print(f"\nWeekly Revenue Stats:")
print(f"  Mean:   ${weekly['revenue_won'].mean():,.0f}")
print(f"  Median: ${weekly['revenue_won'].median():,.0f}")
print(f"  Min:    ${weekly['revenue_won'].min():,.0f}")
print(f"  Max:    ${weekly['revenue_won'].max():,.0f}")

# Show historical revenue chart
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['revenue_won'],
    mode='lines+markers', name='Weekly Revenue',
    line=dict(color='#4299e1', width=2),
    marker=dict(size=4)
))
fig.update_layout(
    title='Historical Weekly Revenue (Won Deals)',
    xaxis_title='Week', yaxis_title='Revenue ($)',
    template='plotly_dark', height=400
)
fig.show()

In [ ]:
# Step 2: Run the full forecast pipeline
# This function handles: Prophet training, XGBoost training,
# ensemble blending, walk-forward CV, and confidence bands
print("Running hybrid forecast pipeline (this may take a minute)...")
results = run_forecast_pipeline(df, forecast_weeks=12)

forecast_df = results['forecast']['forecast_df']
historical_df = results['forecast']['historical_df']
cv_results = results['forecast']['cv_results']
shap_analysis = results['shap_analysis']

print(f"\n✅ Forecast generated: {len(forecast_df)} weeks ahead")
print(f"\nWalk-Forward Cross-Validation Results:")
for key, val in cv_results.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# Step 3: Visualize Forecast
fig = go.Figure()

# Historical
fig.add_trace(go.Scatter(
    x=historical_df['week_start'], y=historical_df['revenue_won'],
    mode='lines+markers', name='Historical Revenue',
    line=dict(color='#4299e1', width=2),
    marker=dict(size=5, opacity=0.7)
))

# Forecast
fig.add_trace(go.Scatter(
    x=forecast_df['week_start'], y=forecast_df['ensemble_forecast'],
    mode='lines+markers', name='Hybrid Forecast',
    line=dict(color='#48bb78', width=3, dash='dot'),
    marker=dict(size=6)
))

# Confidence band
fig.add_trace(go.Scatter(
    x=pd.concat([forecast_df['week_start'], forecast_df['week_start'][::-1]]),
    y=pd.concat([forecast_df['ensemble_upper'], forecast_df['ensemble_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(72, 187, 120, 0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    name='80% Confidence Band'
))

# Show model components if available
if 'prophet_forecast' in forecast_df.columns:
    fig.add_trace(go.Scatter(
        x=forecast_df['week_start'], y=forecast_df['prophet_forecast'],
        mode='lines', name='Prophet (Trend)',
        line=dict(color='orange', width=1, dash='dash'), opacity=0.5
    ))

fig.add_trace(go.Scatter(
    x=forecast_df['week_start'], y=forecast_df['xgb_forecast'],
    mode='lines', name='XGBoost (Recency)',
    line=dict(color='purple', width=1, dash='dash'), opacity=0.5
))

fig.update_layout(
    title='Hybrid Revenue Forecast: Prophet + XGBoost Ensemble (12 Weeks)',
    xaxis_title='Week', yaxis_title='Revenue ($)',
    template='plotly_dark', height=550,
    legend=dict(orientation='h', y=1.05),
    hovermode='x unified'
)
fig.show()

In [ ]:
# Forecast Data Table
display_df = forecast_df[['week_start', 'ensemble_forecast', 'ensemble_lower', 'ensemble_upper']].copy()
display_df.columns = ['Week', 'Forecast ($)', 'Lower Bound ($)', 'Upper Bound ($)']
display_df['Week'] = display_df['Week'].dt.strftime('%Y-%m-%d')
for col in ['Forecast ($)', 'Lower Bound ($)', 'Upper Bound ($)']:
    display_df[col] = display_df[col].apply(lambda x: f'${x:,.0f}')

print("\n12-Week Revenue Forecast:")
display_df

### Forecast Interpretation

- **The ensemble** (green dotted line) is more stable than either model alone
- **Prophet** (orange) captures the overall trend and any quarterly patterns
- **XGBoost** (purple) reacts faster to recent pipeline shifts
- **Confidence bands** widen as we go further out — beyond 8 weeks, treat the forecast as directional, not precise

The MAPE from walk-forward CV gives a realistic error bound for planning purposes.

---
## Engine D: Pipeline Anomaly Detection

### Problem Definition

*"Are any pipeline metrics deviating significantly from historical norms? Which specific segments are behaving unusually?"*

### Method: Statistical Z-Score Analysis

1. Compute monthly values for a chosen metric (e.g., win rate)
2. Calculate rolling mean and std (4-month window)
3. Z-score = `(value - rolling_mean) / rolling_std`
4. Flag as anomaly if `|Z-score| > threshold`

We also run **segment-level** anomaly detection to find which regions or industries are behaving unusually.

In [ ]:
# Step 1: Overall Anomaly Detection
df['closed_year_month'] = df['closed_date'].dt.to_period('M').astype(str)

monthly = df.groupby('closed_year_month').agg(
    win_rate=('is_won', 'mean'),
    deal_count=('deal_id', 'count'),
    avg_deal_size=('deal_amount', 'mean'),
    avg_cycle=('sales_cycle_days', 'mean')
).reset_index()

monthly['win_rate'] = (monthly['win_rate'] * 100).round(2)

# Rolling statistics for anomaly detection
window = 4  # 4-month rolling window
threshold = 2.0  # Z-score threshold

metric = 'win_rate'
monthly['rolling_mean'] = monthly[metric].rolling(window=window, min_periods=2).mean()
monthly['rolling_std'] = monthly[metric].rolling(window=window, min_periods=2).std()
monthly['z_score'] = (monthly[metric] - monthly['rolling_mean']) / monthly['rolling_std'].clip(lower=0.01)
monthly['is_anomaly'] = monthly['z_score'].abs() > threshold

anomalies = monthly[monthly['is_anomaly']]
print(f"Anomaly Detection Results (metric: {metric}, threshold: {threshold}σ):")
print(f"  Total months analyzed: {len(monthly)}")
print(f"  Anomalies detected: {len(anomalies)}")
if len(anomalies) > 0:
    print(f"\n  Anomalous months:")
    for _, row in anomalies.iterrows():
        direction = '📈 Above' if row['z_score'] > 0 else '📉 Below'
        print(f"    {row['closed_year_month']}: {metric}={row[metric]:.1f}% (Z={row['z_score']:.2f}) {direction} normal")

In [ ]:
# Anomaly Detection Visualization
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    x=monthly['closed_year_month'], y=monthly[metric],
    mode='lines+markers', name=f'{metric}',
    line=dict(color='#4299e1', width=2)
))

# Rolling mean
fig.add_trace(go.Scatter(
    x=monthly['closed_year_month'], y=monthly['rolling_mean'],
    mode='lines', name='Rolling Average',
    line=dict(color='#ecc94b', width=2, dash='dash')
))

# Upper/lower bounds
fig.add_trace(go.Scatter(
    x=monthly['closed_year_month'],
    y=monthly['rolling_mean'] + threshold * monthly['rolling_std'],
    mode='lines', name=f'+{threshold}σ',
    line=dict(color='rgba(255,75,75,0.3)', dash='dot')
))
fig.add_trace(go.Scatter(
    x=monthly['closed_year_month'],
    y=monthly['rolling_mean'] - threshold * monthly['rolling_std'],
    mode='lines', name=f'-{threshold}σ',
    line=dict(color='rgba(255,75,75,0.3)', dash='dot'),
    fill='tonexty', fillcolor='rgba(255,75,75,0.05)'
))

# Anomaly markers
if len(anomalies) > 0:
    fig.add_trace(go.Scatter(
        x=anomalies['closed_year_month'], y=anomalies[metric],
        mode='markers', name='🚨 Anomaly',
        marker=dict(size=15, color='red', symbol='x', line=dict(width=2, color='white'))
    ))

fig.update_layout(
    title=f'Anomaly Detection: {metric} (Z-Score Method, {threshold}σ threshold)',
    xaxis_title='Month', yaxis_title=metric,
    template='plotly_dark', height=500
)
fig.show()

In [ ]:
# Step 2: Segment-Level Anomaly Detection
# Which regions/industries are behaving unusually compared to their own history?

def detect_segment_anomalies(df, segment_col, metric_col='is_won', recent_months=3):
    """Detect segments whose recent performance deviates from their historical baseline."""
    df = df.copy()
    df['month'] = df['closed_date'].dt.to_period('M')
    all_months = df['month'].unique()
    all_months = sorted(all_months)
    
    cutoff = all_months[-recent_months] if len(all_months) > recent_months else all_months[0]
    
    results = []
    for seg in df[segment_col].unique():
        seg_df = df[df[segment_col] == seg]
        historical = seg_df[seg_df['month'] < cutoff]
        recent = seg_df[seg_df['month'] >= cutoff]
        
        if len(historical) < 10 or len(recent) < 5:
            continue
        
        hist_rate = historical[metric_col].mean() * 100
        recent_rate = recent[metric_col].mean() * 100
        hist_std = historical.groupby('month')[metric_col].mean().std() * 100
        
        change = recent_rate - hist_rate
        z_score = change / hist_std if hist_std > 0 else 0
        
        severity = 'High' if abs(z_score) > 2.5 else 'Medium' if abs(z_score) > 2.0 else 'Low' if abs(z_score) > 1.5 else 'Normal'
        
        results.append({
            'segment': seg,
            'historical_rate': round(hist_rate, 1),
            'recent_rate': round(recent_rate, 1),
            'change_pp': round(change, 1),
            'z_score': round(z_score, 2),
            'severity': severity
        })
    
    return pd.DataFrame(results).sort_values('z_score')

# Run for regions
region_anomalies = detect_segment_anomalies(df, 'region')
print("Segment Anomaly Detection — REGIONS:")
print("=" * 70)
for _, row in region_anomalies.iterrows():
    emoji = '🔴' if row['severity'] == 'High' else '🟡' if row['severity'] == 'Medium' else '🟢' if row['severity'] == 'Low' else '⚪'
    print(f"  {emoji} {row['segment']:15s} | Hist: {row['historical_rate']:.1f}% → Recent: {row['recent_rate']:.1f}% | Δ{row['change_pp']:+.1f}pp | Z={row['z_score']:+.2f} | {row['severity']}")

# Run for industries
industry_anomalies = detect_segment_anomalies(df, 'industry')
print(f"\nSegment Anomaly Detection — INDUSTRIES:")
print("=" * 70)
for _, row in industry_anomalies.iterrows():
    emoji = '🔴' if row['severity'] == 'High' else '🟡' if row['severity'] == 'Medium' else '🟢' if row['severity'] == 'Low' else '⚪'
    print(f"  {emoji} {row['segment']:15s} | Hist: {row['historical_rate']:.1f}% → Recent: {row['recent_rate']:.1f}% | Δ{row['change_pp']:+.1f}pp | Z={row['z_score']:+.2f} | {row['severity']}")

In [ ]:
# Segment Anomaly Visualization
all_anomalies = pd.concat([
    region_anomalies.assign(type='Region'),
    industry_anomalies.assign(type='Industry')
])
all_anomalies = all_anomalies[all_anomalies['severity'] != 'Normal'].sort_values('z_score')

if len(all_anomalies) > 0:
    colors = all_anomalies['change_pp'].apply(lambda x: '#f56565' if x < 0 else '#48bb78')
    
    fig = go.Figure(go.Bar(
        x=all_anomalies['change_pp'],
        y=all_anomalies['segment'] + ' (' + all_anomalies['type'] + ')',
        orientation='h',
        marker_color=colors,
        text=all_anomalies.apply(lambda r: f"{r['change_pp']:+.1f}pp ({r['severity']})", axis=1),
        textposition='outside'
    ))
    fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.5)
    fig.update_layout(
        title='Segment-Level Anomalies: Win Rate Change (pp)',
        xaxis_title='Change from Historical (percentage points)',
        template='plotly_dark', height=max(400, len(all_anomalies)*40)
    )
    fig.show()
else:
    print("No significant segment-level anomalies detected.")

---
## Bonus: SkyRalph — Conversational CRM Agent

### Architecture Overview

SkyRalph is a **LangGraph ReAct agent** that allows CROs to ask natural language questions and get data-backed answers.

```
User Question → LLM (Gemini 2.5 Flash) → Tool Selection → Execution → Response
                         ↑                                    ↓
                         └────── Observe ←──── Tool Output ───┘
```

### Agent Tools

| Tool | Function | Example Query |
|------|----------|---------------|
| `get_key_metrics_summary` | PQS, WRE, SMI, DVI | "How's the business doing?" |
| `get_revenue_forecast` | Full forecast pipeline | "Forecast next 8 weeks" |
| `explain_win_rate_trends` | EDA insights | "Why are we losing more deals?" |
| `analyze_sales_data` | Ad-hoc pandas queries | "List top 5 reps by revenue" |

The `analyze_sales_data` tool uses `create_pandas_dataframe_agent` from LangChain, which dynamically generates and executes Python code to answer arbitrary questions about the data. It can also produce Plotly charts rendered inline in the chat.

**Note:** The agent requires a `GOOGLE_API_KEY` environment variable to function. See the `pages/3_💬_CRM_Agent.py` and `src/agent/` directory for the full implementation.

In [ ]:
# Show the agent's system prompt and tool definitions
from src.agent.graph import SYSTEM_PROMPT
from src.agent.tools import get_key_metrics_summary, get_revenue_forecast, explain_win_rate_trends, analyze_sales_data

print("SkyRalph System Prompt:")
print("=" * 60)
print(SYSTEM_PROMPT)

print("\n\n" + "=" * 60)
print("Available Tools:")
print("=" * 60)
tools = [get_key_metrics_summary, get_revenue_forecast, explain_win_rate_trends, analyze_sales_data]
for tool in tools:
    print(f"\n🔧 {tool.name}")
    print(f"   {tool.description[:120]}...")

In [ ]:
# Demo: Call one of the agent's tools directly
# (This doesn't require the LLM — it's a direct function call)
import json

print("Calling get_key_metrics_summary() directly...")
print("=" * 60)
summary_json = get_key_metrics_summary.invoke({})
summary = json.loads(summary_json)

print(f"\n📊 Overall Performance:")
overall = summary.get('overall', {})
print(f"  Total Deals:  {overall.get('total_deals')}")
print(f"  Won Deals:    {overall.get('won_deals')}")
print(f"  Win Rate:     {overall.get('win_rate_pct')}%")
print(f"  Total Revenue: ${overall.get('total_revenue', 0):,.0f}")

print(f"\n🔑 Pipeline Qualification:")
pqs = summary.get('pqs', {})
print(f"  PQS Score:    {pqs.get('pqs_score')}/100 [{pqs.get('pqs_category')}]")

print(f"\n📈 Win Rate Elasticity:")
wre = summary.get('wre', {})
print(f"  Elasticity:   {wre.get('elasticity')}")
print(f"  Sweet Spot:   {wre.get('sweet_spot_range')}")

---
## Summary: All Four Engines Working Together

| Engine | Tier | What It Provides |
|--------|------|------------------|
| **B: Win/Loss Classifier** | Diagnostic | *Why* deals win or lose (SHAP drivers) |
| **C: Revenue Forecast** | Predictive | *What's coming* next quarter (with confidence) |
| **D: Anomaly Detection** | Alerting | *What's unusual* right now (Z-score flags) |
| **Bonus: SkyRalph** | Interactive | *Ask anything* in plain English |

Together, they form a complete decision intelligence system:

1. **Monday morning:** Check the Anomaly Alerts → Are there any fires to put out?
2. **Pipeline review:** Check the Win Rate Drivers → Which factors should we optimize?
3. **Quarterly planning:** Check the Revenue Forecast → What should we budget for?
4. **Ad-hoc questions:** Ask SkyRalph → Get instant, data-backed answers